# 3. FREAK (Fast Retina Keypoint) con Bolsa de Palabras Visuales (BoVW)
### Clasificación de Categorías de Imágenes: Manzanas, Perros y Personas

Este cuaderno implementa la solución completa de **Image Detection / Classification** utilizando el descriptor binario bio-inspirado **FREAK**:
- **Entrada**: Una imagen cualquiera.
- **Salida**: La clase predicha a la que pertenece la imagen (`apples`, `dogs` o `people`).

#### ¿Qué es FREAK y cómo se aplica para clasificar imágenes?
1. **Descriptor Bio-inspirado**: FREAK emula la fóvea de la retina humana mediante círculos concéntricos de campos receptivos, con alta densidad en el centro y decaimiento hacia la periferia.
2. **Descriptor Binario de 512 bits (64 bytes)**: Cada bit es el resultado de una comparación de intensidades filtradas entre pares de campos receptivos.
3. **Detector Rápido FAST**: Dado que FREAK es exclusivamente un descriptor (no detecta puntos clave), se acopla con el detector ultra-rápido de esquinas FAST (`cv2.FastFeatureDetector_create`).
4. **Bolsa de Palabras Visuales (BoVW)**: Los descriptores binarios se agrupan en un vocabulario visual ($k = 150$) mediante K-Means para generar histogramas de frecuencias normalizados por imagen y clasificar con SVM.



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from dataset import load_split
from freak_pipeline import FREAKBoVWPipeline

def show_image(img, title=None):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if len(img.shape) == 3 else img
    plt.figure(figsize=(5, 5))
    plt.imshow(img_rgb)
    plt.axis('off')
    if title:
        plt.title(title, fontsize=12, fontweight='bold')
    plt.show()



## Paso 1: Carga de Datos y Detección de Puntos Clave FAST + FREAK


In [ ]:
train_imgs, train_lbls, classes, _ = load_split("train")
test_imgs, test_lbls, _, _ = load_split("test")

# Detector FAST + Extractor FREAK
fast = cv2.FastFeatureDetector_create(threshold=25, nonmaxSuppression=True)
freak = cv2.xfeatures2d.FREAK_create()

sample_img = train_imgs[10]
gray = cv2.cvtColor(sample_img, cv2.COLOR_BGR2GRAY)
kps = fast.detect(gray, None)
kps = sorted(kps, key=lambda k: k.response, reverse=True)[:300]
kps, descs = freak.compute(gray, kps)

print(f"Puntos clave FAST detectados: {len(kps)}")
print(f"Dimensión del descriptor FREAK: {descs.shape} (uint8, 64 bytes = 512 bits)")

img_kps = cv2.drawKeypoints(sample_img, kps, None, color=(0, 255, 255), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
show_image(img_kps, title="FAST Keypoints + FREAK Bio-Inspired Retinal Descriptors")



## Paso 2: Creación del Vocabulario Visual y Entrenamiento de FREAK BoVW


In [ ]:
freak_bovw = FREAKBoVWPipeline(detector_type="fast", max_keypoints=300, vocab_size=150)
print("Construyendo vocabulario visual FREAK y entrenando clasificador SVM...")
info = freak_bovw.train(train_imgs, train_lbls, C=1.0)
print(f"Entrenamiento completado en {info['train_time_sec']:.2f} segundos!")
print(f"Vocabulario visual: {info['vocab_size']} palabras visuales")



## Paso 3: Evaluación en el Conjunto de Prueba


In [ ]:
eval_res = freak_bovw.evaluate(test_imgs, test_lbls)
print(f"Exactitud en Prueba FREAK BoVW (Accuracy): {eval_res['accuracy'] * 100:.2f}%")
print(f"Latencia promedio por imagen:              {eval_res['avg_time_per_image_ms']:.2f} ms/imagen
")
print("Reporte de Clasificación por Clase:")
print(classification_report(test_lbls, eval_res['predictions'], target_names=freak_bovw.classes_))

# Matriz de confusión
cm = np.array(eval_res["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Oranges")
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, fontsize=11)
ax.set_yticklabels(classes, fontsize=11)
plt.xlabel("Clase Predicha", fontsize=11)
plt.ylabel("Clase Real", fontsize=11)
plt.title("Matriz de Confusión FREAK + BoVW", fontsize=12, fontweight='bold')
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="red" if i != j else "black", fontweight='bold')
plt.colorbar(im)
plt.tight_layout()
plt.show()



## Paso 4: Demostración de Clasificación en Imágenes Individuales


In [ ]:
import random

for _ in range(3):
    idx = random.randrange(len(test_imgs))
    img = test_imgs[idx]
    true_label = test_lbls[idx]
    
    pred_label = freak_bovw.predict([img])[0]
    show_image(img, title=f"Real: {true_label.upper()} | Predicción FREAK: {pred_label.upper()}")



## Paso 5: Guardado del Modelo


In [ ]:
freak_bovw.save_model("freak_bovw_model.pkl")
sz = Path("freak_bovw_model.pkl").stat().st_size / (1024 * 1024)
print(f"Modelo guardado: freak_bovw_model.pkl ({sz:.2f} MB)")

